# InscetCount
### Automated Digitization and Specimen Counting from Entomological Collection Drawer Images

This repository contains the code for an automated pipeline for digitising and inventorying entomological museum collections from whole-drawer images. The pipeline combines a YOLOv11 object detection model with optical character recognition (OCR) to detect specimens, genus and species labels within drawer photographs, and outputs structured specimen counts per taxon as a CSV file.

In [1]:
import os
import re
from datetime import datetime
import cv2 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as img
from pathlib import Path
from PIL import Image
from IPython.display import clear_output
import imutils
from ultralytics import YOLO
from paddleocr import PaddleOCR
from natsort import natsorted
import pandas as pd
import logging


In [2]:
date = datetime.now().strftime("%d-%m-%Y")
print(date)

30-03-2026


In [ ]:
def Make_jpg(tif_path, save_path):
    """
    Converts tif images to jpg format and saves file in the new path provided.
    """
    
    img = Image.open(tif_path)
    img_jpg = img.convert("RGB")

    img_jpg.save(save_path,
        format="JPEG",quality=100, optimize=True,
        subsampling=0)   


def Add_qrcode_scale_whitebal(image_path, save_path, text, card_loc):
    """
    To each image, its unique drawer ID and a scale bar is added to the image on the top right corner.
    The scale bar added to the image is a approximate measurement based on the size of the color card boxes. 
    Images are then whitebalanced using the color card in the image.
    The image is then saved in the new folder path provided.
    """
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    #########
    ## 1. Add drawer ID on the top right corner on the image
    font_scale = h * 0.0005
    thickness = max(2, int(h * 0.001))
    font = cv2.FONT_HERSHEY_SIMPLEX

    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    padding = int(th * 0.4)

    margin = int(h * 0.02) 
    x = w - tw - 2 * padding - margin
    y = 0 + padding + margin

    if text:
        cv2.rectangle(img,(x, y - th - padding),(x + tw + 2 * padding, y + baseline),(255, 255, 255),-1)
        cv2.putText(img, text,(x + padding, y),font,font_scale,(0, 0, 0),thickness,cv2.LINE_AA)

    #########
    ## 2. Add scale on the top right corner on the image, below the drawer ID
    img_with_scale = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    cord1, cord2, cord3, cord4 = card_loc
    card = gray[cord1:cord2, cord3:cord4] 

    lower = 190
    ret, thresh = cv2.threshold(card, lower, 255, cv2.THRESH_BINARY)

    cnts = cv2.findContours(thresh, mode=cv2.RETR_LIST, method=cv2.CHAIN_APPROX_NONE)
    cnts = imutils.grab_contours(cnts)
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:1]

    white_x, white_y, white_w, white_h = cv2.boundingRect(cnts[0])

    real_cm = 1.6
    px_per_cm = white_w / real_cm
    scale_cm = 3.0
    scale_px = int(scale_cm * px_per_cm)


    padding = 500
    x_start = w - scale_px - padding
    y_start = 0 + padding

    x_end = x_start + scale_px
    y_end = y_start 

    
    thickness=40 
    cv2.rectangle(img_with_scale,(x_start, y_start - thickness // 2),(x_end,   y_start + thickness // 2),(255, 255, 255),-1)
    cv2.putText(img_with_scale,"3 cm",(x_start, y_start - 40),font,5,(255, 255, 255),7,cv2.LINE_AA)


    #########
    ## 3. Whitebalancing

    h = int(white_h/2) + 2
    y = int(white_y + h)
    h = 5

    w = int(white_w/2) - 2
    x = int(white_x + w)
    w = 5

    card = img_with_scale[cord1:cord2, cord3:cord4]
    WHITE_BOX = card[y:y+h, x:x+w]
    
    v0 = round(242/int(np.average(WHITE_BOX[:,:,0])),4) 
    v1 = round(243/int(np.average(WHITE_BOX[:,:,1])),4)
    v2 = round(243/int(np.average(WHITE_BOX[:,:,2])),4)
            
    (B, G, R) = cv2.split(img_with_scale)
    
    B_l_fun = lambda ele: ele * v0 
    G_l_fun = lambda ele: ele * v1
    R_l_fun = lambda ele: ele * v2
    clamp = lambda ele:np.clip(ele, 0, 255)
    
    B = B_l_fun(B)   
    G = G_l_fun(G)  
    R = R_l_fun(R)  
        
    merged = cv2.merge([B, G, R])
    merged = clamp(merged)

    img8 = merged.astype('uint8')
 
    cv2.imwrite(save_path, img8)


def Crop_pad(Image_path, save_path, crop_dim):
    """
    Crops a specific region of the image and pads it to create a square output.

    The cropping focuses on isolating the drawer region while removing irrelevant
    parts of the image. Since all drawers are aligned to the top-left corner,
    approximate coordinates (crop_dim) are used for cropping.

    After cropping, the image is padded with black pixels on the top and bottom
    to ensure a square aspect ratio.

    Optional resizing is included (commented out) if a fixed output size is required.
    """

    cord1, cord2, cord3, cord4 = crop_dim
    image = cv2.imread(Image_path)

    image = image[cord1:cord2, cord3:cord4]  ## Drawer coordinates
            
    h, w, _ = image.shape
    total_padding = max(0, w - h)
    pad_top = total_padding // 2
    pad_bottom = total_padding - pad_top

    padded_image = cv2.copyMakeBorder(image, pad_top, pad_bottom, 0, 0, cv2.BORDER_CONSTANT, value=(0, 0, 0))

    # Optional: Resize image to a fixed resolution (e.g., 640x640)
    # down_size = (640, 640)
    # padded_image = cv2.resize(padded_image, down_size, interpolation= cv2.INTER_AREA)

    cv2.imwrite(save_path, padded_image)


def Specimen_count(file_path,img_folder,out_folder):
    """
    Processes YOLO annotation txt file to extract taxonomic information and count specimens.

    Workflow:
    1. Parse annotation file and organize objects into drawer columns.
       - Columns are sorted from left-to-right based on x-coordinate.
       - Objects (specimens, genus labels, species labels) are assigned to their
         respective columns and sorted top-to-bottom, then left-to-right.

    2. Perform OCR on detected genus and species label regions.
       - Image regions corresponding to label bounding boxes are cropped.
       - OCR is applied to extract text, which is appended to the annotation txt file.

    3. Aggregate results.
       - Counts total genus labels, species labels, and specimen instances per drawer.
       - Groups specimens by genus and species.
       - summary dictionary for the drawer is returned 
    """

    ########
    ### Assign objects to their containing columns, sort columns left-to-right and sort objects within each column top-to-bottom then left-to-right.
    name = os.path.basename(file_path).split(".txt")[0]
    img_path = f"{img_folder}/{name}.jpg"  
    output_file = f"{out_folder}/{name}_new.txt"


    with open(file_path, 'r') as file:
        lines = file.readlines()

    annotations = []
    for line in lines:
        parts = line.strip().split()
        annotations.append({
            'class_id': int(parts[0]),
            'x': float(parts[1]),
            'y': float(parts[2]),
            'w': float(parts[3]),
            'h': float(parts[4]),
            'full_line': f"{parts[0]} {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n"
            })
        parts = parts[5:] 

    ### Separate columns (class = 3) left-to-right by x_center 
    sorted_columns = sorted([a for a in annotations if a['class_id'] == 3], key=lambda b: b['x'])
    ### Rest of the objects (class = 0, 1, 2)  
    object_classes={0, 1, 2}
    objects = [a for a in annotations if a['class_id'] in object_classes]  


    ### Assign each object to the column it belongs to (fallback: nearest column by x distance)
    col_objects = {i: [] for i in range(len(sorted_columns))} 
    for obj in objects:
        for i, col in enumerate(sorted_columns):
            if (abs(obj['x'] - col['x']) <= col['w'] / 2 and abs(obj['y'] - col['y']) <= col['h'] / 2):
                col_objects[i].append(obj)
                break

    ### Build sorted label list: Column first, then its objects sorted top-to-bottom / left-to-right
    sorted_labels = []
    for i, col in enumerate(sorted_columns):
        sorted_labels.append(col['full_line'])
        for obj in sorted(col_objects[i], key=lambda o: (o['y'], o['x'])):
            sorted_labels.append(obj['full_line'])

    ########
    image = cv2.imread(img_path)
    h, w, _ = image.shape

    updated_labels = []
    for label in sorted_labels:
        parts = label.split()
        object_class, x_center, y_center, width, height = map(float, parts)
        object_class = int(object_class)
        if object_class == 0 or object_class == 1:
            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)
            cropped = image[y1:y2, x1:x2]

            result = OCR.ocr(cropped)

            if result[0] is not None:
                text = " ".join([line[1][0] for line in result[0]])
            else:
                text = "unreadable"

            updated_label = label.strip() + " " + text + "\n"
            updated_labels.append(updated_label)
        else:
            updated_labels.append(label)

    # Write the updated labels with OCR text to the output file
    with open(output_file, 'a') as file:
        file.writelines(updated_labels)


    ########

    genus = None
    species = None
    specimen_count = 0

    total_genus = 0
    total_species = 0
    total_specimens = 0

    sp_list = []
        
    for label in updated_labels:
        parts = label.split()
        object_class = int(parts[0])
        
        # -------- GENUS --------
        if object_class == 0:
            total_genus += 1
            # Close previous species BEFORE changing genus
            if species is not None:
                if specimen_count == 0:
                    specimen_count = 1
                sp_list.append((genus, species, specimen_count))
                specimen_count = 0
                species = None
                
            genus = " ".join(parts[5:])

        # -------- SPECIES --------
        elif object_class == 1:
            total_species += 1
            # Close previous species
            if species is not None:
                if specimen_count == 0:
                    specimen_count = 1
                sp_list.append((genus, species, specimen_count))
                specimen_count = 0
            # If no genus seen yet --> Unknown
            if genus is None:
                genus = "Unknown_genus"
            species = " ".join(parts[5:])

        # -------- SPECIMENS --------
        elif object_class == 2:
            total_specimens += 1
            specimen_count += 1

    # Append final block
    if species is not None:
        if specimen_count == 0:
            specimen_count = 1
        sp_list.append((genus, species, specimen_count))


    summary = {
        "Drawer_ID": name,
        "total_genus_labels": total_genus,
        "total_species_labels": total_species,
        "total_specimens": total_specimens,
        "species_list": "; ".join([str(s) for s in sp_list])
    }


    return summary



def Preprocess_and_count(working_dir, tif_dir, model, num, card_dim, drawer_dim):

    """
    Workflow for processing specimen drawer images.

    This function converts TIFF images to JPG, applies preprocessing
    (QR code, scale, white balance, cropping, and padding), runs object
    detection, performs OCR on detected labels, and returns specimen
    count summaries for each image.

    """

    jpg_dir = Path(f"{working_dir}/jpg")
    jpg_qr_scale_white_dir = Path(f"{working_dir}/jpg_qr_scale_white")          
    jpg_crop_pad_dir = Path(f"{working_dir}/jpg_crop_pad")
    model_pred_dir = Path(f"{working_dir}/model_pred_out")

    for dir in [jpg_dir, jpg_qr_scale_white_dir, jpg_crop_pad_dir, model_pred_dir]:
        dir.mkdir(exist_ok=True)   

    all_summaries = []
    all_sp_lists = []

    for tif_path in tif_dir.glob("*.tif"):

        jpg_path = f"{jpg_dir}/{tif_path.stem}.jpg"  
        jpg_qr_scale_white = f"{jpg_qr_scale_white_dir}/{tif_path.stem}.jpg"
        jpg_crop_pad_path = f"{jpg_crop_pad_dir}/{tif_path.stem}.jpg"

        ##### 1. Convert to JPG
        ## only if the jpg doesn't already exist, otherwise it will skip to next steps.
        if not os.path.exists(jpg_path):
            Make_jpg(tif_path, jpg_path)
            

        ##### 2. Qrcode, sacel, Whitebalance and crop
        ## Qrcode and scale are added using function Add_qrcode_scale, only if set to True, otherwise only white balance and crop is done.
        if not os.path.exists(jpg_qr_scale_white):
            if num:
                num_str = f"{num:06d}"
                text = f"SMFCOLD{num_str}"
                num += 1
            else:
                text = ""  # skip addition of QR number
            img = Add_qrcode_scale_whitebal(jpg_path, jpg_qr_scale_white, text, card_dim)


        ##### 3. Crop and pad and resize
        ## only if the cropped, padded and reszie if it doesn't already exist, otherwise it will skip to next steps.
        if not os.path.exists(jpg_crop_pad_path):
            Crop_pad(jpg_qr_scale_white, jpg_crop_pad_path, drawer_dim)

        
        ##### 4. Model prediction
        results = model.predict(jpg_crop_pad_path, 
                save=True, 
                conf=0.3, # defalut = 0.25
                show_labels=False,
                show_boxes = True,
                save_txt = True,
                #show_conf = False,
                project=model_pred_dir,
                exist_ok=True,
                #save_dir=out_dir,
                #save_crop=True,
                verbose=False
                )
        
        results = model.predict(jpg_crop_pad_path)
            
        ##### 5. Specimen count and OCR
        txt_file = Path(f"{model_pred_dir}/predict/labels/{tif_path.stem}.txt")
        summary = Specimen_count(txt_file, jpg_crop_pad_dir, model_pred_dir)
        all_summaries.append(summary)

    return all_summaries


###########################################################################

### Test on images

In [ ]:
logging.getLogger('ppocr').setLevel(logging.ERROR)
OCR = PaddleOCR(
    lang='en',
    det_model_dir="/home/PaddleOCR/PP-OCRv4_server_det_infer",
    rec_model_dir='/home/PaddleOCR/PP-OCRv4_server_rec_infer',
    det_limit_side_len=1536, # default = 960 . Set higher for better detection of smaller text
    precision = 'fp32', # default = 'fp32'. Set to 'fp16' for faster inference
    det_db_box_thresh = 0.4,  # default = 0.6
    det_db_thresh=0.2  # default = 0.3. Set lower for better detection of faint text
    )


working_dir = "/home/digi/drawers2/Carabidae_test_drawers"  
tif_dir = Path(f"{working_dir}/original_tif") ## Folder is within the the working_dir


model = YOLO("/home/digi/drawers2/ultralytics/runs/detect/yolo11m_drawer_train_b32_run3_640_17-03-2026_final/weights/best.pt")
card_dim = (3500, 6000, 8500, 9100) # (3500:5500, 8500:9100)
drawer_dim = (0, 6500, 0, 8500) # [0:6500, 0:8500]
num=None  ### set None when you don't want to add QR code, otherwise give a starting number for the QR code, e.g. num=1 will add SMFCOLD000001, SMFCOLD000002, etc.


In [ ]:
all_summaries = Preprocess_and_count(working_dir=working_dir, tif_dir=tif_dir, model=model, num=num, card_dim=card_dim, drawer_dim=drawer_dim)

df_summaries = pd.DataFrame(all_summaries)
# df_summaries.to_csv(Path(f"{working_dir}/Drawer_summary_out.csv"), index=False) 